# Pipeline BERTopic — Analyse des Motifs d'Annulation des RDV

Objectif :
1. Charger uniquement les RDV annulés depuis PostgreSQL/Supabase.
2. Nettoyer légèrement les commentaires sans détruire leur sens métier.
3. Découvrir automatiquement les motifs avec BERTopic.
4. Réduire le nombre de topics si nécessaire.
5. Afficher les mots-clés + exemples par topic pour faciliter le nommage métier.
6. Exporter les données, le modèle et le dictionnaire de motifs.



## Installation des dépendances (si nécessaire)

In [1]:
!pip install bertopic sentence-transformers umap-learn hdbscan pandas psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 45.5 MB/s eta 0:00:00


## Imports et Configuration

In [2]:
from __future__ import annotations

import json
import logging
import os
import re
import unicodedata
from pathlib import Path
import psycopg2

import pandas as pd
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer

### Paramètres de configuration

In [3]:
# Configuration des colonnes et valeurs
COMMENT_COLUMN = "commentaire"
STATUS_COLUMN = "status"
CANCELLED_VALUE = "Annulé"

# Modèle d'embedding
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

# Paramètres BERTopic
MIN_TOPIC_SIZE = 15
N_TOPICS = 15

RANDOM_STATE = 42

# Répertoires de sortie
OUTPUT_DIR = Path("outputs")
MODEL_DIR = Path("models")

OUTPUT_DATA = OUTPUT_DIR / "rdv_avec_motifs_bertopic.csv"
OUTPUT_TOPICS = OUTPUT_DIR / "topics_bertopic_details.csv"
OUTPUT_HTML = OUTPUT_DIR / "bertopic_visualisation.html"
OUTPUT_MAPPING = MODEL_DIR / "noms_motifs.json"
OUTPUT_MODEL = MODEL_DIR / "bertopic_motifs"

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)

### Listes de nettoyage du texte

In [4]:
# Mots très fréquents mais peu discriminants
WORDS_TO_REMOVE = {
    "mr",
    "mme",
    "monsieur",
    "madame",
    "rdv",
    "confirmé",
    "confirme",
    "pv",
    "pac",
    "chauffage",
    "facture",
    "client",
}

# Stopwords français
FRENCH_STOPWORDS = {
    "a", "ai", "aie", "aient", "as", "au", "aura", "avec",
    "ce", "cela", "ces", "cet", "cette", "comme",
    "dans", "de", "des", "du",
    "elle", "elles", "en", "es", "est", "et", "eux",
    "il", "ils",
    "je",
    "la", "le", "les", "leur", "leurs", "lui",
    "ma", "mais", "me", "mes", "moi", "mon",
    "ne", "nos", "notre", "nous",
    "on", "ou",
    "par", "pas", "pour",
    "que", "quel", "quelle", "quelles", "quels", "qui",
    "sa", "sans", "se", "sera", "ses", "son", "sont", "sur",
    "ta", "te", "tes", "toi", "ton", "tous", "tout", "toute", "toutes",
    "un", "une",
    "vos", "votre", "vous",
    "y",
}

# Patterns de nettoyage
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
NON_LETTER_PATTERN = re.compile(r"[^a-zàâäæçéèêëîïôœùûüÿ0-9' -]+")
MULTISPACE_PATTERN = re.compile(r"\s+")

## Connexion à la Base de Données

In [5]:
db_url = 'postgresql://postgres.####################3.pooler.supabase.com:5432/postgres'
querry="""
    SELECT
        id AS rdv_id,
        status,
        commentaire
    FROM rendez_vous
    WHERE status = 'Annulé';
    """
conn = psycopg2.connect(db_url)
df = pd.read_sql_query(querry,conn)
print(df)


                                    rdv_id  status  \
0     a568c8b7-2dfe-44b6-8e19-f00e3f9ed205  Annulé   
1     784a7f73-ae94-4c2b-8c6d-4b4fa48ec956  Annulé   
2     ff25a99f-1789-4dd0-b408-b4b9dc6de368  Annulé   
3     dbab7997-c245-4620-9510-537481de5e8c  Annulé   
4     3dfb9506-df73-4501-a735-0ebebb58d303  Annulé   
...                                    ...     ...   
2405  62b4e2d6-6946-4e05-b764-b91e9177c90c  Annulé   
2406  2998078b-9e16-47f6-9fa2-e2f703bfe589  Annulé   
2407  a9789971-5346-4054-bb2d-3d1d68883e12  Annulé   
2408  e76c0006-5719-4474-95c2-2b4b9fecb66f  Annulé   
2409  4244b575-2521-4976-ae41-463650c87771  Annulé   

                                            commentaire  
0     RAP pour confirmation présence Mr, verouillage...  
1     RDV ANNULER SON CONJOINT ET PAS DISPO IL TRAVA...  
2     Rdv PV pris avec Mr, Fact Elec 320€/mois, Crit...  
3     Mr a des difficultés financières et est connai...  
4     MAISON  +15 ANS  \nFACTURE ELECT +150 \nFIOUL ...  
...

## Nettoyage du Texte

In [6]:
def remove_accents_for_matching(text: str) -> str:
    """Retourne une version sans accents uniquement pour certains contrôles."""
    normalized = unicodedata.normalize("NFD", text)
    return "".join(
        char for char in normalized
        if unicodedata.category(char) != "Mn"
    )


def clean_comment(text: object) -> str:
    """
    Nettoyage léger orienté NLP.

    On évite un nettoyage agressif :
      - les accents sont conservés ;
      - les chiffres sont conservés car '85 ans', '5000', etc. peuvent être
        porteurs d'information métier ;
      - la ponctuation est réduite ;
      - les mots génériques sont supprimés comme mots entiers.
    """
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    if not text:
        return ""

    text = URL_PATTERN.sub(" ", text)

    # Apostrophes typographiques -> apostrophe simple.
    text = text.replace("'", "'")

    text = NON_LETTER_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()

    tokens = []
    for token in text.split():
        # Comparaison sans accent pour éviter des variantes comme confirmé /
        # confirme dans la liste de nettoyage.
        token_for_matching = remove_accents_for_matching(token)

        if token_for_matching in WORDS_TO_REMOVE:
            continue

        tokens.append(token)

    return " ".join(tokens)


def prepare_texts(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Nettoie les commentaires et retire les commentaires vides."""
    result = df.copy()

    result["commentaire_clean"] = result[COMMENT_COLUMN].apply(clean_comment)

    mask = result["commentaire_clean"].str.len() > 0
    texts = result.loc[mask, "commentaire_clean"].tolist()

    logger.info(
        "Commentaires non vides après nettoyage : %d/%d",
        len(texts),
        len(result),
    )

    if len(texts) < MIN_TOPIC_SIZE:
        raise ValueError(
            f"Seulement {len(texts)} commentaires exploitables. "
            f"MIN_TOPIC_SIZE={MIN_TOPIC_SIZE} est trop élevé."
        )

    return result, texts

## Modèle BERTopic

In [7]:
def build_topic_model(min_topic_size: int = MIN_TOPIC_SIZE) -> BERTopic:
    """Construit BERTopic avec des paramètres reproductibles."""
    logger.info("Chargement du modèle d'embedding : %s", EMBEDDING_MODEL)

    embedding_model = SentenceTransformer(EMBEDDING_MODEL)

    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_STATE,
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=min_topic_size,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )

    vectorizer_model = CountVectorizer(
        stop_words=list(FRENCH_STOPWORDS),
        ngram_range=(1, 2),
        min_df=2,
    )

    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        language="multilingual",
        calculate_probabilities=False,
        verbose=True,
    )


def fit_topics(
    model: BERTopic,
    texts: list[str],
    nr_topics: int | None = N_TOPICS,
) -> tuple[BERTopic, list[int]]:
    """Entraîne BERTopic puis réduit les topics si demandé."""
    logger.info("Entraînement BERTopic sur %d commentaires...", len(texts))

    topics, _ = model.fit_transform(texts)

    logger.info(
        "Topics bruts détectés : %d",
        len(model.get_topic_info()),
    )

    if nr_topics is not None:
        logger.info(
            "Réduction automatique vers environ %d topics...",
            nr_topics,
        )
        model.reduce_topics(texts, nr_topics=nr_topics)
        topics = model.topics_

    logger.info(
        "Topics après réduction : %d",
        len(model.get_topic_info()),
    )

    return model, list(topics)

## Analyse et Export des Résultats

In [8]:
def build_topic_table(model: BERTopic) -> pd.DataFrame:
    """
    Crée une table exploitable pour le dashboard/jury :
    topic, nombre, nom automatique et mots-clés.
    """
    info = model.get_topic_info().copy()

    keywords = []

    for topic_id in info["Topic"]:
        if topic_id == -1:
            keywords.append("Bruit / non classé")
            continue

        topic_words = model.get_topic(topic_id)

        if topic_words:
            keywords.append(
                ", ".join(word for word, _ in topic_words[:10])
            )
        else:
            keywords.append("")

    info["keywords"] = keywords

    return info


def print_topic_analysis(model: BERTopic, texts: list[str]) -> None:
    """Affiche les topics et quelques exemples pour aider au nommage métier."""
    info = build_topic_table(model)

    print("\n" + "=" * 80)
    print("TOPICS DÉCOUVERTS")
    print("=" * 80)

    for _, row in info.iterrows():
        topic_id = int(row["Topic"])
        count = int(row["Count"])

        print(f"\nTopic {topic_id} | {count} commentaires")
        print(f"Mots-clés : {row['keywords']}")

        if topic_id == -1:
            continue

        representative_docs = model.get_representative_docs(topic_id)

        if representative_docs:
            print("Exemples :")
            for doc in representative_docs[:3]:
                print(f"  - {doc[:180]}")


def assign_topics(
    df: pd.DataFrame,
    topics: list[int],
) -> pd.DataFrame:
    """Réinjecte les topics dans le DataFrame original."""
    result = df.copy()
    result["motif_topic"] = -1

    mask = result["commentaire_clean"].str.len() > 0

    if mask.sum() != len(topics):
        raise ValueError(
            "Le nombre de topics ne correspond pas au nombre de "
            "commentaires non vides."
        )

    result.loc[mask, "motif_topic"] = topics
    result["motif_topic"] = result["motif_topic"].astype(str)

    return result


def build_recap(
    df: pd.DataFrame,
    motif_column: str = "motif_nom",
) -> pd.DataFrame:
    """Répartition des motifs parmi les RDV annulés."""
    total = len(df)

    if total == 0:
        return pd.DataFrame(columns=["Nombre", "% des annulations"])

    recap = (
        df.groupby(motif_column, dropna=False)
        .size()
        .to_frame("Nombre")
        .sort_values("Nombre", ascending=False)
    )

    recap["% des annulations"] = (
        recap["Nombre"] / total * 100
    ).round(1)

    return recap

## Nommage des Topics par Métier

In [9]:
def apply_business_labels(
    df: pd.DataFrame,
    model: BERTopic,
    labels: dict[str, str],
) -> pd.DataFrame:
    """
    Transforme les IDs de topics en noms métier.

    Les labels doivent être renseignés APRÈS inspection des mots-clés et
    exemples affichés par print_topic_analysis().
    """
    result = df.copy()

    def label_topic(topic_id: str) -> str:
        if topic_id in labels:
            return labels[topic_id]

        if topic_id == "-1":
            return "Non classé"

        try:
            topic_words = model.get_topic(int(topic_id))
            if topic_words:
                return f"Topic {topic_id} — {topic_words[0][0]}"
        except Exception:
            pass

        return f"Topic {topic_id}"

    result["motif_nom"] = result["motif_topic"].apply(label_topic)

    return result

## Sauvegarde des Résultats

In [10]:
def save_outputs(
    df: pd.DataFrame,
    model: BERTopic,
    labels: dict[str, str],
) -> None:
    """Sauvegarde les données, le modèle, les détails et la visualisation."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    topic_info = build_topic_table(model)

    df.to_csv(OUTPUT_DATA, index=False, encoding="utf-8-sig")
    topic_info.to_csv(OUTPUT_TOPICS, index=False, encoding="utf-8-sig")

    with open(OUTPUT_MAPPING, "w", encoding="utf-8") as file:
        json.dump(labels, file, ensure_ascii=False, indent=2)

    logger.info("Sauvegarde du modèle BERTopic...")
    model.save(
        str(OUTPUT_MODEL),
        serialization="safetensors",
        save_ctfidf=True,
        save_embedding_model=True,
    )

    try:
        fig = model.visualize_topics()
        fig.write_html(str(OUTPUT_HTML))
    except Exception as exc:
        logger.warning("Visualisation non générée : %s", exc)

    logger.info("Exports terminés.")
    logger.info("Données : %s", OUTPUT_DATA)
    logger.info("Topics  : %s", OUTPUT_TOPICS)
    logger.info("Modèle  : %s", OUTPUT_MODEL)
    logger.info("Mapping : %s", OUTPUT_MAPPING)

## Exécution du Pipeline Principal

### Étape 1 : Configuration de la base de données

Définissez la variable d'environnement DATABASE_URL avant de continuer :

### Étape 2 : Charger les données et nettoyer

In [11]:
# Nettoyer et préparer les textes
df, texts = prepare_texts(df)


### Étape 3 : Construire et entraîner le modèle BERTopic

In [12]:
# Construire le modèle
model = build_topic_model()

# Entraîner et réduire les topics
model, topics = fit_topics(model, texts, nr_topics=N_TOPICS)

print("\n✓ Modèle BERTopic entraîné et optimisé")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-09-16 21:52:38,378 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/75 [00:00<?, ?it/s]

2026-09-16 21:53:42,271 - BERTopic - Embedding - Completed ✓
2026-09-16 21:53:42,274 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-16 21:54:09,101 - BERTopic - Dimensionality - Completed ✓
2026-09-16 21:54:09,102 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-16 21:54:09,221 - BERTopic - Cluster - Completed ✓
2026-09-16 21:54:09,230 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-16 21:54:09,355 - BERTopic - Representation - Completed ✓
2026-09-16 21:54:09,553 - BERTopic - Topic reduction - Reducing number of topics
2026-09-16 21:54:09,570 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-16 21:54:09,702 - BERTopic - Representation - Completed ✓
2026-09-16 21:54:09,705 - BERTopic - Topic reduction - Reduced number of topics from 48 to 15



✓ Modèle BERTopic entraîné et optimisé


### Étape 4 : Assigner les topics aux commentaires

In [13]:
# Ajouter les topics au DataFrame
df = assign_topics(df, topics)

print("✓ Topics assignés aux commentaires")

✓ Topics assignés aux commentaires


### Étape 5 : Analyser les topics et identifier les motifs

In [14]:
# Afficher l'analyse détaillée des topics
print_topic_analysis(model, texts)

print("\n💡 Inspectez les mots-clés et les exemples ci-dessus pour identifier les motifs métier.")


TOPICS DÉCOUVERTS

Topic -1 | 523 commentaires
Mots-clés : Bruit / non classé

Topic 0 | 758 commentaires
Mots-clés : ok, financement, présence, techniques ok, techniques, ok financement, expliqué présence, expliqué, financement expliqué, fact
Exemples :
  - avec fact élec 1030 critères techniques ok financement expliqué présence du couple validée durée étude et doc ok
  - avec fact élec 1010 critères techniques ok financement expliqué présence du couple validée durée étude et doc ok
  - avec fact élec 2450 critères techniques ok financement expliqué présence du couple validée durée étude et doc ok

Topic 1 | 221 commentaires
Mots-clés : élect, critères documents, documents, critères, documents présence, couple ok, mois critères, présence couple, couple, ok
Exemples :
  - par élect de 126 critères documents et présence couple ok
  - par élect de 177 mois critères documents et présence couple ok
  - par élect de 104 mois critères documents et présence couple ok

Topic 2 | 135 commentai

### Étape 6 : Appliquer les noms métier aux topics

Remplissez le dictionnaire `business_labels` avec vos motifs métier identifiés :

In [15]:
# Remplissez ce dictionnaire selon vos motifs métier
business_labels = {
     "0": "Client a changé d'avis ",
     "1": "Eligible mais refuse",
     "2": "Concurrent",
     "3": "Refus de crédit",
     "4": "Lancer un rappel",
     "5": "Client garde équipement Gaz",
     "6": "Client n'est pas intéressé",
     "7": "SMS de rappel rdv date ultérieure",
     "8": "Client déjà équipé",
     "9": "Désacord du couple",
     "10":"infos habitation/ps de motif",
     "11":"Problème de toiture",
     "12":"fausse barbe",
     "13": "Problème age client/maison",
     "-1": "Non classé",
}

# Appliquer les labels métier
df = apply_business_labels(
    df,
    model,
    business_labels,
)

print("✓ Labels métier appliqués")

✓ Labels métier appliqués


### Étape 7 : Afficher le récapitulatif des motifs

In [16]:
recap = build_recap(df, "motif_nom")

print("\n" + "=" * 80)
print("RÉPARTITION DES MOTIFS D'ANNULATION")
print("=" * 80)
print(recap)


RÉPARTITION DES MOTIFS D'ANNULATION
                                   Nombre  % des annulations
motif_nom                                                   
Client a changé d'avis                758               31.5
Non classé                            535               22.2
Eligible mais refuse                  221                9.2
Concurrent                            135                5.6
Refus de crédit                       116                4.8
Lancer un rappel                      113                4.7
Client garde équipement Gaz           110                4.6
Client n'est pas intéressé            101                4.2
SMS de rappel rdv date ultérieure      70                2.9
Client déjà équipé                     60                2.5
Désacord du couple                     52                2.2
infos habitation/ps de motif           48                2.0
Problème de toiture                    34                1.4
fausse barbe                           30       

### Étape 8 : Exporter les résultats

In [17]:
# Sauvegarder tous les résultats
save_outputs(df, model, business_labels)

print("\n✓ Pipeline terminé avec succès !")
print(f"\nFichiers générés :")
print(f"  - Données  : {OUTPUT_DATA}")
print(f"  - Topics  : {OUTPUT_TOPICS}")
print(f"  - Modèle  : {OUTPUT_MODEL}")
print(f"  - Mapping : {OUTPUT_MAPPING}")


✓ Pipeline terminé avec succès !

Fichiers générés :
  - Données  : outputs/rdv_avec_motifs_bertopic.csv
  - Topics  : outputs/topics_bertopic_details.csv
  - Modèle  : models/bertopic_motifs
  - Mapping : models/noms_motifs.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')